In [2]:
# solve_reinos.py
# Requisitos atendidos:
# - Matriz 6x6, estados: ouro (G), trigo (W), vazio (.)
# - Reinos NÃO precisam ser contínuos
# - NÃO precisam usar todos os quadrados
# - 4 reinos precisam ser IDÊNTICOS em forma (mesma “máscara”)
# - NÃO pode ser espelhado (apenas rotações 0/90/180/270 + translação)
# - Cada reino: 9 células, com 2 ouros + 2 trigos (ajustável)

from __future__ import annotations
from itertools import combinations
from typing import List, Tuple, Dict, Optional
from PIL import Image, ImageDraw

In [ ]:

N = 6  # 6x6

# -----------------------------
# Utilidades de bitmask (36 bits)
# -----------------------------
def rc_to_idx(r: int, c: int):
    return r * N + c

def idx_to_rc(i: int):
    return divmod(i, N)

def mask_to_coords(mask: int):
    coords = []
    for i in range(N * N):
        if (mask >> i) & 1:
            coords.append(idx_to_rc(i))
    return coords

def normalize(coords: List[Tuple[int, int]]):
    min_r = min(r for r, _ in coords)
    min_c = min(c for _, c in coords)
    out = [(r - min_r, c - min_c) for r, c in coords]
    out.sort()
    return out

def coords_key(coords: List[Tuple[int, int]]):
    # coords já normalizadas e ordenadas
    return "|".join(f"{r},{c}" for r, c in coords)

# rotações (sem espelho)
def rot0(p):   return ( p[0],  p[1])
def rot90(p):  return ( p[1], -p[0])
def rot180(p): return (-p[0], -p[1])
def rot270(p): return (-p[1],  p[0])

def canonical_signature(mask: int) -> str:
    coords = mask_to_coords(mask)
    rots = []
    for fn in (rot0, rot90, rot180, rot270):
        rotated = [fn(p) for p in coords]
        rots.append(coords_key(normalize(rotated)))
    return min(rots)

def count_resources(mask: int, grid: List[str]) -> Tuple[int, int]:
    g = w = 0
    for i in range(N * N):
        if (mask >> i) & 1:
            if grid[i] == "G":
                g += 1
            elif grid[i] == "W":
                w += 1
    return g, w

# -----------------------------
# Geração de candidatos (com poda por contagem)
# -----------------------------
def generate_candidates(
    grid: List[str],
    size: int = 9,
    need_g: int = 2,
    need_w: int = 2
) -> List[int]:
    gold = [i for i, v in enumerate(grid) if v == "G"]
    wheat = [i for i, v in enumerate(grid) if v == "W"]
    empty = [i for i, v in enumerate(grid) if v == "."]

    need_e = size - need_g - need_w
    if need_e < 0:
        return []

    cands = []
    for g2 in combinations(gold, need_g):
        for w2 in combinations(wheat, need_w):
            for e5 in combinations(empty, need_e):
                mask = 0
                for i in (*g2, *w2, *e5):
                    mask |= (1 << i)
                cands.append(mask)
    return cands

# -----------------------------
# Busca por 4 máscaras disjuntas com mesma assinatura
# -----------------------------
def find_k_disjoint(masks: List[int], k: int = 4) -> Optional[List[int]]:
    # backtracking simples (funciona bem porque a lista por assinatura costuma ser pequena)
    def dfs(start: int, picked: List[int], used: int) -> Optional[List[int]]:
        if len(picked) == k:
            return picked
        for i in range(start, len(masks)):
            m = masks[i]
            if (m & used) == 0:
                res = dfs(i + 1, picked + [m], used | m)
                if res:
                    return res
        return None

    return dfs(0, [], 0)

def solve(
    grid_lines: List[str],
    size: int = 9,
    need_g: int = 2,
    need_w: int = 2,
    kingdoms: int = 4
) -> Optional[Dict]:
    # grid_lines: 6 strings com 6 chars cada (G/W/.)
    grid = [ch for row in grid_lines for ch in row.strip()]
    if len(grid) != 36:
        raise ValueError("Grid inválido: precisa ter 6 linhas de 6 caracteres (total 36).")

    candidates = generate_candidates(grid, size=size, need_g=need_g, need_w=need_w)

    groups: Dict[str, List[int]] = {}
    for m in candidates:
        sig = canonical_signature(m)
        groups.setdefault(sig, []).append(m)

    # tenta achar qualquer assinatura com >= kingdoms máscaras disjuntas
    for sig, masks in groups.items():
        if len(masks) < kingdoms:
            continue
        found = find_k_disjoint(masks, k=kingdoms)
        if found:
            return {"signature": sig, "masks": found, "grid": grid, "grid_lines": grid_lines}

    return None

# -----------------------------
# Renderização simples em PNG
# -----------------------------
def render_solution_png(
    grid_lines: List[str],
    masks: List[int],
    out_path: str = "solucao.png",
    cell: int = 70,
    pad: int = 20
):
    # cores por reino
    colors = [
        (255,  0,  0),
        (  0,120,255),
        (255,200,  0),
        (  0,180, 80),
    ]

    W = pad * 2 + N * cell
    H = pad * 2 + N * cell
    img = Image.new("RGB", (W, H), (245, 245, 245))
    d = ImageDraw.Draw(img)

    # grade
    for r in range(N + 1):
        y = pad + r * cell
        d.line([(pad, y), (pad + N * cell, y)], fill=(30, 30, 30), width=2)
    for c in range(N + 1):
        x = pad + c * cell
        d.line([(x, pad), (x, pad + N * cell)], fill=(30, 30, 30), width=2)

    # desenha recursos
    for r in range(N):
        for c in range(N):
            ch = grid_lines[r][c]
            x0 = pad + c * cell
            y0 = pad + r * cell
            x1 = x0 + cell
            y1 = y0 + cell
            if ch == "G":
                d.ellipse([x0+18, y0+18, x1-18, y1-18], outline=(0,0,0), width=3)
                d.text((x0+cell//2-10, y0+cell//2-10), "G", fill=(0,0,0))
            elif ch == "W":
                d.rectangle([x0+18, y0+18, x1-18, y1-18], outline=(0,0,0), width=3)
                d.text((x0+cell//2-10, y0+cell//2-10), "W", fill=(0,0,0))

    # bordas de cada reino (não precisa ser contínuo; desenha contorno por célula)
    for k, mask in enumerate(masks):
        col = colors[k % len(colors)]
        for i in range(N * N):
            if (mask >> i) & 1:
                r, c = idx_to_rc(i)
                x0 = pad + c * cell
                y0 = pad + r * cell
                x1 = x0 + cell
                y1 = y0 + cell
                # borda grossa em volta da célula
                d.rectangle([x0+3, y0+3, x1-3, y1-3], outline=col, width=6)

    img.save(out_path)


In [5]:

# -----------------------------
# Exemplo de uso
# -----------------------------
if __name__ == "__main__":
    # Preencha com seu tabuleiro real (6 linhas de 6 chars: G/W/.)
    # Exemplo (placeholder):
    grid_lines = [
        "W.W.G.",
        "G.G..W",
        "..G..W",
        ".W...G",
        "W.W.G.",
        ".GG.W."
]

    result = solve(grid_lines, size=9, need_g=2, need_w=2, kingdoms=4)

    if not result:
        print("Nenhuma solução encontrada com esses parâmetros.")
    else:
        print("Solução encontrada!")
        print("Assinatura:", result["signature"])
        for i, m in enumerate(result["masks"], 1):
            g, w = count_resources(m, result["grid"])
            print(f"Reino {i}: mask={m} | ouro={g} trigo={w}")
        render_solution_png(grid_lines, result["masks"], out_path="solucao.png")
        print("Imagem salva em: solucao.png")


Solução encontrada!
Assinatura: 0,0|0,1|0,2|0,3|0,4|0,5|1,0|1,3|1,4
Reino 1: mask=1663 | ouro=2 trigo=2
Reino 2: mask=260480 | ouro=2 trigo=2
Reino 3: mask=435945472 | ouro=2 trigo=2
Reino 4: mask=68283269120 | ouro=2 trigo=2
Imagem salva em: solucao.png
